## Проверка результатов подбора параметров с помощью optuna

In [ ]:
import json

with open('/wrk/main/masked experiments/optuna_padim_18_masks.json', 'r') as f:
    data = json.load(f)

filtered_data = {key: value for key, value in data.items() if "error" not in value}

data_items = sorted(filtered_data.items(), key=lambda x: x[1]["metrics"]['pixel_F1Score'], reverse=True)

data_items[:2]

Пример вывода данных:

[('34',  
  {'trial': 34,  
   'params': {'layers': ['layer3', 'layer4'],  
    'n_features': 310,  
    'sensitivity': 0.50013230476432907,  
    'resize_size': 512},  
   'metrics': {'image_Accuracy': 0.8365734705352783,  
    'image_Precision': 0.8985731727256775,  
    'image_Recall': 0.7412347058029175,  
    'image_F1Score': 0.8175838059844971,  
    'image_AUROC': 0.8390214705352783,  
    'pixel_Accuracy': 0.8612358655662537,  
    'pixel_F1Score': 0.6336121306838989,  
    'pixel_AUROC': 0.8243266138381958},  
   'result': 0.6336121306838989}),  
    
 ('102',  
  {'trial': 102,  
   'params': {'layers': ['layer4'],  
    'n_features': 340,  
    'sensitivity': 0.50387197148173,  
    'resize_size': 512},  
   'metrics': {'image_Accuracy': 0.8398805589294434,  
    'image_Precision': 0.812493,  
    'image_Recall': 0.8841659412956238,  
    'image_F1Score': 0.8476924833908081,  
    'image_AUROC': 0.8340320589294434,  
    'pixel_Accuracy': 0.8608519082984924,  
    'pixel_F1Score': 0.6329044623947144,  
    'pixel_AUROC': 0.8362359633789062},  
   'result': 0.6329044623947144})]   

In [ ]:
import json

with open('/wrk/main/masked experiments/optuna_padim_50_masks.json', 'r') as f:
    data = json.load(f)

filtered_data = {key: value for key, value in data.items() if "error" not in value}

data_items = sorted(filtered_data.items(), key=lambda x: x[1]["metrics"]['pixel_F1Score'], reverse=True)

data_items[:5]

## Пример воспроизведения обучения с наилучшими параметрами

In [ ]:
from anomalib.engine import Engine
import torch
from anomalib.models.image.padim import Padim
from torchvision import models
import os
from anomalib.data import Folder
from pytorch_lightning import seed_everything
from torchmetrics.classification import BinaryAccuracy, BinaryPrecision, BinaryRecall, BinaryF1Score, BinaryAUROC
from anomalib.metrics import Evaluator, AnomalibMetric
from anomalib.post_processing import PostProcessor
from anomalib.pre_processing import PreProcessor
from torchvision.transforms.v2 import Resize

torch.cuda.empty_cache()
torch.set_float32_matmul_precision('medium')
seed_everything(42, workers=True)

# -----------------
#      Metrics
# -----------------

class Accuracy(AnomalibMetric, BinaryAccuracy):
    pass

class Precision(AnomalibMetric, BinaryPrecision):
    pass

class Recall(AnomalibMetric, BinaryRecall):
    pass

class F1Score(AnomalibMetric, BinaryF1Score):
    pass

class AUROC(AnomalibMetric, BinaryAUROC):
    pass

image_Accuracy = Accuracy(
    fields=["pred_label", "gt_label"],
    prefix="image_"
)

image_Precision = Precision(
    fields=["pred_label", "gt_label"],
    prefix="image_"
)

image_Recall = Recall(
    fields=["pred_label", "gt_label"],
    prefix="image_"
)

image_F1Score = F1Score(
    fields=["pred_label", "gt_label"],
    prefix="image_"
)

image_AUROC = AUROC(
    fields=["pred_label", "gt_label"],
    prefix="image_"
)

pixel_Accuracy = Accuracy(
    fields=["pred_mask", "gt_mask"],
    prefix="pixel_"
)

pixel_F1Score = F1Score(
    fields=["pred_mask", "gt_mask"],
    prefix="pixel_"
)

pixel_AUROC = AUROC(
    fields=["pred_mask", "gt_mask"],
    prefix="pixel_"
)

evaluator = Evaluator(test_metrics=[image_Accuracy, image_Precision, image_Recall, image_F1Score, image_AUROC, pixel_Accuracy, pixel_F1Score, pixel_AUROC]) 

post_processor = PostProcessor(
    image_sensitivity=0.50013230476432907,
    pixel_sensitivity=0.50013230476432907
)

pre_processor = PreProcessor(transform=Resize(size=(512,512)))

# -----------------
#       Data
# -----------------

train_dataset = Folder(
    name="main",
    root="/wrk/main/masked experiments/data",
    normal_dir="train",
    abnormal_dir="defect_val",
    mask_dir="masks_val",
    normal_test_dir="normal_val",
    train_batch_size=10,
    eval_batch_size=10,
    num_workers=4,
    seed=42
)

# -----------------
#      Models
# -----------------

weights = torch.load("/wrk/CNN_weights/resnet18.pth", map_location='cuda')
custom_backbone = models.resnet18()
custom_backbone.load_state_dict(weights)
custom_backbone.eval()

if custom_backbone:
    print("Local weights loaded successfully")

model = Padim(
    backbone=custom_backbone,
    layers=['layer3', 'layer4'],
    n_features=310,
    pre_trained=False,
    evaluator=evaluator,
    post_processor=post_processor,
    pre_processor=pre_processor
)

device = 'cuda'
model = model.to(device)

engine = Engine(
                accelerator='cuda', 
                enable_progress_bar=True
                )

engine.train(model=model, datamodule=train_dataset)

print(engine.trainer.callback_metrics)